# Priority Classifier v3 — Approccio Ibrido (LinearSVC + LLM)

**ZHC — Zucchetti Healthcare | TicketClassifier**

## Strategia
Il LinearSVC v2 raggiunge già macro F1 = 0.8451.  
L'obiettivo non è sostituirlo, ma **rafforzarlo** sui casi incerti:

| Situazione | Approccio |
|---|---|
| decision_function margin alto | LinearSVC diretto (veloce, economico) |
| margin basso / ambiguità P1-P2 | LLM con regole PO16 + few-shot |
| P4 (rara, spesso confusa) | Sempre LLM come secondo parere |

## Features del ticket in input
- `oggetto` — titolo del ticket (sempre valorizzato)
- `descrizione` — testo pulito (99.8% coverage)
- `priorita_iniziale_cliente` — P1/P2/P3/P4 impostata dal cliente sul portale ZConnect

---

## STEP 1 — Import e configurazione percorsi

In [1]:
import os
import re
import json
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Literal, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# ─── Percorsi di progetto ────────────────────────────────────────────────────
BASE_DIR  = Path(r'C:\Users\matteo.segatto\Desktop\TicketClassifier')
MODEL_DIR = BASE_DIR / 'modelli' / 'priority_v2'
DATA_DIR  = BASE_DIR / 'data'
EMB_DIR   = BASE_DIR / 'embeddings'
OUT_DIR   = BASE_DIR / 'data'

load_dotenv(BASE_DIR / '.env')

print('BASE_DIR:', BASE_DIR)
print('MODEL_DIR exists:', MODEL_DIR.exists())
print('API key OpenAI:', 'trovata' if os.getenv('OPENAI_API_KEY') else 'MANCANTE')

# Se usi Claude invece di OpenAI:
# print('API key Anthropic:', 'trovata' if os.getenv('ANTHROPIC_API_KEY') else 'MANCANTE')

BASE_DIR: C:\Users\matteo.segatto\Desktop\TicketClassifier
MODEL_DIR exists: True
API key OpenAI: trovata


## STEP 2 — Carica modello LinearSVC v2 e metadata

In [2]:
# Carica il classificatore SVC già addestrato
svc = joblib.load(MODEL_DIR / 'classificatore_svc.pkl')

with open(MODEL_DIR / 'metadata.json') as f:
    meta = json.load(f)

CLASSI           = meta['classi']           # ['P1', 'P2', 'P3', 'P4']
KEYWORD_URGENZA  = meta['keyword_urgenza']  # ['urgente', 'immediato', ...]
EMBEDDING_MODEL  = meta['modello_embedding']  # 'intfloat/multilingual-e5-base'
E5_PREFIX        = meta['prefisso_e5']       # 'query: '

print('Modello caricato:', type(svc).__name__)
print('Classi:', CLASSI)
print('Macro F1 su test set:', meta['macro_f1_test'])
print('Accuracy su test set:', meta['accuracy_test'])

Modello caricato: LinearSVC
Classi: ['P1', 'P2', 'P3', 'P4']
Macro F1 su test set: 0.8451
Accuracy su test set: 0.8821


## STEP 3 — Carica encoder embedding (E5) e OHE priorità

Il modello usa 2 feature concatenate:
1. Embedding E5 768d (testo ticket)
2. One-hot encoding priorità iniziale cliente (4 valori: P1/P2/P3/P4)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse as sp

# ─── Modello embedding ───────────────────────────────────────────────────────
print('Carico E5 (può richiedere qualche secondo)...')
e5_model = SentenceTransformer(EMBEDDING_MODEL)
print('E5 caricato.')

# ─── OHE priorità iniziale cliente ──────────────────────────────────────────
# Ricarica lo stesso encoder usato in training.
# Se il file non esiste, lo ricostruiamo con le stesse categorie.
ohe_path = MODEL_DIR / 'ohe_encoder.pkl'
if ohe_path.exists():
    ohe = joblib.load(ohe_path)
    print('OHE caricato da file.')
else:
    ohe = OneHotEncoder(
        categories=[['P1', 'P2', 'P3', 'P4']],
        sparse_output=True,
        handle_unknown='ignore'
    )
    # Fit sulle 4 categorie fisse (non serve dataset)
    ohe.fit([['P1'], ['P2'], ['P3'], ['P4']])
    print('OHE ricreato (file non trovato — ok se usi le stesse categorie).')

def build_features(oggetto: str,
                   descrizione: str,
                   priorita_iniziale: str = 'P3') -> np.ndarray:
    """Costruisce il vettore feature per un singolo ticket."""
    # 1. Testo completo
    testo = f"{oggetto or ''} {descrizione or ''}".strip()

    # 2. Embedding E5 (con prefisso 'query: ')
    emb = e5_model.encode(
        E5_PREFIX + testo,
        normalize_embeddings=True
    ).reshape(1, -1)  # shape (1, 768)

    # 3. OHE priorità iniziale cliente
    p_init = priorita_iniziale if priorita_iniziale in ['P1','P2','P3','P4'] else 'P3'
    ohe_vec = ohe.transform([[p_init]])  # sparse (1, 4)

    # 4. Concatenazione: [768d | 4d] = 772d (stesso schema del training)
    X = np.hstack([
        emb,
        ohe_vec.toarray()
    ])
    return X  # shape (1, 772)

# Test rapido
X_test = build_features(
    'Il sistema non funziona',
    'La piattaforma è completamente inaccessibile da questa mattina',
    priorita_iniziale='P1'
)
print('Feature shape:', X_test.shape)  # deve essere (1, 772)

## STEP 4 — Classificatore SVC con confidenza calibrata

`LinearSVC` non ha `.predict_proba()`, ma ha `.decision_function()` che restituisce
la distanza dal margine. Usiamo la distanza per stimare la confidenza:
- **margin alto** → il modello è sicuro → output diretto
- **margin basso** → ambiguità → passiamo all'LLM

Normalizziamo con softmax sui decision scores per avere una pseudo-probabilità
confrontabile con l'output strutturato dell'LLM.

In [ ]:
from scipy.special import softmax

def predict_svc_with_confidence(X: np.ndarray) -> tuple[str, float, dict]:
    """
    Predice la priorità con il LinearSVC e stima la confidenza.

    Returns:
        classe_predetta: 'P1', 'P2', 'P3' o 'P4'
        confidenza:      float in [0, 1] — pseudo-probabilità softmax
        probs:           dict con pseudo-prob per ogni classe
    """
    scores = svc.decision_function(X)[0]  # shape (4,) — una per classe
    probs  = softmax(scores * 2)          # * 2 per accentuare il picco (calibrazione empirica)

    best_idx    = np.argmax(probs)
    classe      = CLASSI[best_idx]
    confidenza  = float(probs[best_idx])
    probs_dict  = {c: float(p) for c, p in zip(CLASSI, probs)}

    return classe, confidenza, probs_dict


# Test rapido
classe, conf, probs = predict_svc_with_confidence(X_test)
print(f'Predizione SVC: {classe} (confidenza: {conf:.1%})')
print('Distribuzione classi:', {k: f'{v:.1%}' for k, v in probs.items()})

## STEP 5 — Definizioni PO16 per il prompt LLM

Queste sono le definizioni **ufficiali** dalla procedura PO16 (Procedura Assistenza SW).
Le usiamo come "ground truth" per il reasoning dell'LLM — il modello non deve inventarsi
le regole, le trova direttamente nel prompt.

In [ ]:
# ─── Definizioni PO16 (versione con focus operativo) ─────────────────────────
PRIORITY_DEFINITIONS = """
P1 — Molto Alta (Blocco Totale):
  Blocco completo e inaccessibilità all'intera piattaforma e relativa operatività.
  TUTTI gli utenti della struttura non riescono a lavorare.
  Si escludono problematiche infrastrutturali esterne (VPN, rete cliente, PC).

P2 — Alta (Blocco Funzionale senza Workaround):
  Blocco di funzionalità essenziali non differibili per cui non sia possibile
  procedere con metodi alternativi (es. terapie, fatturazione, paghe).
  La piattaforma è accessibile ma una funzione critica è inutilizzabile.

P3 — Media (Malfunzionamento con Workaround):
  Malfunzionamento di funzioni essenziali ma gestibile con metodi alternativi
  o supporto operativo. Include: domande procedurali, configurazioni,
  personalizzazioni di elenchi, problemi su singoli utenti/dispositivi.

P4 — Bassa (Impatto Minimo):
  Malfunzionamento che non ostacola il regolare svolgimento del lavoro.
  Richieste evolutive, miglioramenti estetici, anomalie visive con workaround
  immediato, correzioni di dati inseriti erroneamente dall'utente.
"""

# ─── Esempi few-shot — estratti da ticket reali ZHC ──────────────────────────
FEW_SHOT_EXAMPLES = [
    # P1 — ideale: blocco totale multi-dispositivo
    {
        "ticket": "229391",
        "oggetto": "IMPOSSIBILITA' D'ACCESSO",
        "descrizione": ("da questa mattina dalle ore 6:00 circa non riusciamo ad accedere "
                        "a CBA, sia dal PC che dai tablet. Non riusciamo a scrivere le consegne."),
        "priorita_cliente": "P1",
        "priorita_corretta": "P1",
        "reasoning": ("Inaccessibilità totale alla piattaforma da tutti i dispositivi, "
                      "con conseguente fermo operativo dei reparti."),
    },
    # P1 — contrasto: multi-modulo su più strutture = ancora P1
    {
        "ticket": "235829",
        "oggetto": "CONTABILITA' OSPITI AD ES. NON FUNZIONA",
        "descrizione": ("non funzionano i programmi nemmeno in casa CANOSSA, "
                        "sia cartella sanitaria che contabilità ospiti."),
        "priorita_cliente": "P1",
        "priorita_corretta": "P1",
        "reasoning": ("Malfunzionamento trasversale che colpisce più moduli fondamentali "
                      "(Sanitario e Amministrativo) rendendo impossibile il lavoro."),
    },
    # P2 — ideale: blocco funzione clinica critica, nessun workaround
    {
        "ticket": "242930",
        "oggetto": "Terapia al bisogno",
        "descrizione": ("La spunta della terapia al bisogno è accessibile solo quando è attiva "
                        "una terapia cronica. Questo rende impossibile la somministrazione e la "
                        "visualizzazione tramite tablet agli infermieri."),
        "priorita_cliente": "P1",
        "priorita_corretta": "P2",
        "reasoning": ("Blocco di una funzione clinica critica (terapie) senza un modo "
                      "alternativo per registrarla sul momento. La piattaforma è accessibile."),
    },
    # P2 — trappola: URGENTE + scadenza ma problema su singolo record → P2 non P1
    {
        "ticket": "230917",
        "oggetto": "STAMPA BILANCIO",
        "descrizione": ("[URGENTE] Buongiorno, non si riesce a stampare il bilancio in quanto "
                        "segnala errore relativamente ad una fattura. Richiedo collegamento telefonico."),
        "priorita_cliente": "P1",
        "priorita_corretta": "P2",
        "reasoning": ("Sebbene il cliente usi URGENTE e richieda P1, il problema riguarda "
                      "un singolo report/fattura e non l'intera piattaforma. De-escalation a P2."),
    },
    # P3 — ideale: domanda procedurale / supporto operativo
    {
        "ticket": "214951",
        "oggetto": "Problema con l'inserimento dei farmaci",
        "descrizione": ("Non riesco a inserire Pramipexolo con AIC:047845019. "
                        "Vi chiedo di spiegarmi come devo farlo."),
        "priorita_cliente": "P3",
        "priorita_corretta": "P3",
        "reasoning": ("Richiesta di supporto procedurale per l'utilizzo del software, "
                      "non rappresenta un blocco tecnico."),
    },
    # P3 — ideale: richiesta configurazione elenchi
    {
        "ticket": "221582",
        "oggetto": "aggiunta item",
        "descrizione": ("Richiedo l'aggiunta di 'Cintura addominale' nel menu Protezioni "
                        "e una nuova causale di dimissione 'Cambio regime'."),
        "priorita_cliente": "P3",
        "priorita_corretta": "P3",
        "reasoning": ("Richiesta di configurazione di elenchi che non impedisce "
                      "l'attività corrente."),
    },
    # P4 — ideale: correzione errore dati utente, nessun impatto sistema
    {
        "ticket": "244894",
        "oggetto": "Gestione partite/scadenze",
        "descrizione": ("Nel modificare la descrizione in N.partita ho modificato anche "
                        "la situazione sul conto. Grazie per l'aiuto."),
        "priorita_cliente": "P4",
        "priorita_corretta": "P4",
        "reasoning": ("Correzione di un errore di inserimento dati dell'utente che non "
                      "ha impatto sulla stabilità del sistema."),
    },
    # P4 — ideale: anomalia visiva su un solo dispositivo, workaround immediato
    {
        "ticket": "212843",
        "oggetto": "Contatti Assistente Sociale non visibili da Tablet",
        "descrizione": ("Segnalo che da Tablet non è possibile vedere gli assistenti sociali "
                        "tra i contatti, mentre da PC si vedono correttamente."),
        "priorita_cliente": "P4",
        "priorita_corretta": "P4",
        "reasoning": ("Anomalia visiva su un solo tipo di dispositivo con workaround immediato "
                      "(uso del PC). Nessun fermo operativo."),
    },
]

print(f"Definizioni PO16: {len(PRIORITY_DEFINITIONS.strip().splitlines())} righe")
print(f"Esempi few-shot: {len(FEW_SHOT_EXAMPLES)} ({len(FEW_SHOT_EXAMPLES)//2} per classe)")


## STEP 6 — Classificatore LLM con output strutturato (Pydantic)

Usiamo lo stesso pattern del `FewShot_ExampleExtractor` per l'area classifier:
- **Pydantic** per output strutturato garantito
- **Temperatura 0** per riproducibilità
- Il LLM restituisce: `priorita`, `confidenza`, `reasoning`

Funziona sia con OpenAI (GPT-4o-mini) che con Anthropic (Claude Haiku).

In [ ]:
from openai import OpenAI

client_llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ─── Schema output strutturato ───────────────────────────────────────────────
PrioritaType = Literal["P1", "P2", "P3", "P4"]

class ClassificazionePriorita(BaseModel):
    priorita:   PrioritaType = Field(description="Priorità assegnata: P1, P2, P3 o P4")
    confidenza: float        = Field(ge=0.0, le=1.0, description="Confidenza da 0 a 1")
    reasoning:  str          = Field(description="Spiegazione breve (max 2 frasi)")


# ─── Costruzione system prompt ───────────────────────────────────────────────
def build_system_prompt() -> str:
    """Costruisce il system prompt con definizioni PO16, regole operative e few-shot reali."""

    esempi_str = ""
    for ex in FEW_SHOT_EXAMPLES:
        esempi_str += (
            f"\n---"
            f"\nTESTO TICKET: {ex['oggetto']}. {ex['descrizione']}"
            f"\nPRIORITÀ CLIENTE: {ex['priorita_cliente']}"
            f"\n→ PRIORITÀ CORRETTA: {ex['priorita_corretta']}"
            f"\nREASONING: {ex['reasoning']}\n"
        )

    return f"""Sei un assistente specializzato nella classificazione di ticket di assistenza software per Zucchetti Healthcare (ZHC). Il tuo compito è assegnare la priorità corretta a ciascun ticket basandoti sulle definizioni ufficiali della procedura PO16.

## DEFINIZIONI PRIORITÀ (da procedura PO16)
{PRIORITY_DEFINITIONS}

## REGOLE OPERATIVE (applica nell'ordine)

1. DE-ESCALATION OBBLIGATORIA:
   Ignora parole come "URGENTE", "SCADENZA OGGI", "BLOCCATI" se il testo descrive
   un problema su un singolo record, un singolo utente o una funzione secondaria.
   L'urgenza lessicale del cliente NON determina la priorità tecnica.

2. TEST DELLA CONTINUITÀ OPERATIVA:
   Chiedi mentalmente: "Quante persone non riescono a lavorare?"
   - Tutti gli utenti della struttura fermi → P1
   - Un reparto/funzione critica bloccata senza workaround → P2
   - Problema risolvibile con metodo alternativo → P3
   - Un singolo utente, singolo dispositivo o impatto minore → P3 o P4

3. PRIORITÀ CLIENTE NON AFFIDABILE:
   I clienti compilano quasi sempre P1 per errore. Non fidarti della priorità
   iniziale se il testo non descrive un blocco reale, totale e immediato.

4. DISTINZIONE P1 vs P2:
   P1 = inaccessibilità dell'INTERA piattaforma (tutti i moduli, tutti gli utenti).
   P2 = blocco di UNA funzionalità critica senza workaround.
   Se anche un solo modulo funziona o il problema è circoscritto → non è P1.

5. DISTINZIONE P3 vs P4:
   P3 = esiste un malfunzionamento tecnico, ma il lavoro può proseguire.
   P4 = nessun impatto operativo reale (evolutive, estetiche, correzioni dati utente).

## ESEMPI REALI DAL DATASET ZHC
{esempi_str}
Rispondi SOLO con l'oggetto JSON strutturato richiesto. Nessun testo aggiuntivo."""


SYSTEM_PROMPT = build_system_prompt()
print("System prompt generato.")
print(f"Lunghezza: {len(SYSTEM_PROMPT.split())} parole")
print(f"Esempi inclusi: {SYSTEM_PROMPT.count('PRIORITÀ CORRETTA')}")


In [ ]:
def preprocess_testo(oggetto: str, descrizione: str, max_parole: int = 300) -> str:
    """Pulisce e tronca il testo prima di inviarlo all'LLM."""
    testo = f"{oggetto or ''} {descrizione or ''}"
    # Rimuovi HTML
    testo = re.sub(r'<[^>]+>', ' ', testo)
    # Rimuovi firme email e dati di contatto
    testo = re.sub(r'Orario reperibilit.*', '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'Cordiali saluti.*',    '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'Telefono:?\s*[\d\s+]+', '', testo, flags=re.IGNORECASE)
    # Normalizza spazi
    testo = re.sub(r'\s+', ' ', testo).strip()
    # Tronca
    parole = testo.split()
    if len(parole) > max_parole:
        testo = ' '.join(parole[:max_parole]) + '...'
    return testo


def classifica_priorita_llm(oggetto: str,
                             descrizione: str,
                             priorita_iniziale_cliente: str = 'P3') -> ClassificazionePriorita:
    """Classifica la priorità di un ticket usando GPT-4o-mini con few-shot + PO16."""
    testo_pulito = preprocess_testo(oggetto, descrizione)

    user_msg = (
        f'OGGETTO: {oggetto}\n'
        f'DESCRIZIONE: {testo_pulito}\n'
        f'PRIORITÀ IMPOSTATA DAL CLIENTE: {priorita_iniziale_cliente}'
    )

    response = client_llm.beta.chat.completions.parse(
        model='gpt-4o-mini',
        temperature=0,
        max_tokens=150,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        response_format=ClassificazionePriorita,
    )
    return response.choices[0].message.parsed


# ─── Test rapido ─────────────────────────────────────────────────────────────
print('Test LLM su esempio P1 reale...')
risultato = classifica_priorita_llm(
    'SISTEMA COMPLETAMENTE BLOCCATO',
    'Da stamattina nessuno riesce ad entrare nel gestionale. '
    'Abbiamo provato con tutti i PC e tutti gli utenti. '
    'La struttura è completamente ferma, urgente!',
    priorita_iniziale_cliente='P1'
)
print(f'  Priorità:   {risultato.priorita}')
print(f'  Confidenza: {risultato.confidenza:.0%}')
print(f'  Reasoning:  {risultato.reasoning}')

## STEP 7 — Classificatore Ibrido

La classe `HybridPriorityClassifier` combina i due approcci:

```
ticket → LinearSVC → confidenza > soglia? → output diretto
                   →       no           → LLM → output con reasoning
```

**Soglia consigliata**: 0.75 — da calibrare sul validation set.
- Sopra 0.75: ~80% dei ticket → SVC diretto (veloce, gratis)
- Sotto 0.75: ~20% dei ticket → LLM (~0.5 ms latenza, ~$0.0002/ticket)

In [ ]:
from dataclasses import dataclass

@dataclass
class PredizionePriorita:
    """Risultato finale del classificatore ibrido."""
    priorita:    str
    confidenza:  float
    metodo:      str   # 'svc' oppure 'llm'
    reasoning:   Optional[str] = None  # disponibile solo se metodo='llm'
    probs_svc:   Optional[dict] = None  # distribuzione SVC (debug)


class HybridPriorityClassifier:
    """
    Classificatore ibrido LinearSVC + LLM per la priorità dei ticket ZHC.

    Parameters
    ----------
    soglia_confidenza : float
        Soglia sotto la quale il SVC cede il passo all'LLM.
        Default: 0.75 (calibrare sul validation set)
    forza_llm_su_p4 : bool
        Se True, usa sempre l'LLM quando il SVC predice P4.
        P4 è molto rara (1.5% dataset) — conviene validare con LLM.
    """

    def __init__(self, soglia_confidenza: float = 0.75, forza_llm_su_p4: bool = True):
        self.soglia_confidenza = soglia_confidenza
        self.forza_llm_su_p4   = forza_llm_su_p4

    def predict(self,
                oggetto:                  str,
                descrizione:              str,
                priorita_iniziale_cliente: str = 'P3') -> PredizionePriorita:
        """
        Classifica la priorità di un ticket.

        1. Estrae le feature e interroga il LinearSVC
        2. Se confidenza bassa o predizione P4 → usa LLM
        3. Restituisce il risultato con metadati sul percorso usato
        """
        # ── Fast path: LinearSVC ─────────────────────────────────────────────
        X = build_features(oggetto, descrizione, priorita_iniziale_cliente)
        classe_svc, conf_svc, probs_svc = predict_svc_with_confidence(X)

        # Usa SVC direttamente se: confidenza alta E non è P4 (a meno di flag)
        usa_svc = conf_svc >= self.soglia_confidenza
        if self.forza_llm_su_p4 and classe_svc == 'P4':
            usa_svc = False

        if usa_svc:
            return PredizionePriorita(
                priorita   = classe_svc,
                confidenza = conf_svc,
                metodo     = 'svc',
                probs_svc  = probs_svc,
            )

        # ── Slow path: LLM ───────────────────────────────────────────────────
        llm_result = classifica_priorita_llm(oggetto, descrizione, priorita_iniziale_cliente)

        return PredizionePriorita(
            priorita   = llm_result.priorita,
            confidenza = llm_result.confidenza,
            metodo     = 'llm',
            reasoning  = llm_result.reasoning,
            probs_svc  = probs_svc,  # conserviamo i score SVC per debug
        )


# Istanzia il classificatore
clf = HybridPriorityClassifier(soglia_confidenza=0.75, forza_llm_su_p4=True)
print('Classificatore ibrido pronto.')
print(f'Soglia confidenza: {clf.soglia_confidenza:.0%}')
print(f'Forza LLM su P4:   {clf.forza_llm_su_p4}')

## STEP 8 — Test su casi reali

Testiamo sui 4 pattern più comuni dal dataset (vedi EDA.ipynb):
- P1 → P2: 6.129 correzioni (cliente esagera l'urgenza)
- P3 → P2: 4.190 correzioni (cliente sottostima)
- P1 → P3: 2.648 correzioni (domanda ordinaria con P1 per errore)
- P4: rara, spesso confusa con P3

In [ ]:
CASI_TEST = [
    {
        'descrizione_test': 'P1→P2: cliente esagera, blocco modulo non totale',
        'oggetto':          'MODULO PRESENZE BLOCCATO - URGENTISSIMO',
        'descrizione':      'Il modulo Rilevazione Presenze non carica le timbrature di ieri. '
                            'Tutti gli altri moduli funzionano normalmente. '
                            'Abbiamo la chiusura mese domani, è urgentissimo!',
        'p_cliente':        'P1',
        'p_attesa':         'P2',
    },
    {
        'descrizione_test': 'P1→P3: domanda operativa mascherata da urgenza',
        'oggetto':          'Non trovo la funzione stampa cedolini',
        'descrizione':      'Buongiorno, cercavo come si fa a stampare i cedolini in PDF '
                            'dal modulo Gestione Stipendi. Non riesco a trovare il tasto. '
                            'Grazie',
        'p_cliente':        'P1',
        'p_attesa':         'P3',
    },
    {
        'descrizione_test': 'P3→P2: cliente sottostima, bug contabile critico',
        'oggetto':          'Errore calcolo IVA su alcune fatture',
        'descrizione':      'Abbiamo notato che alcune fatture emesse questa settimana '
                            'hanno il calcolo IVA errato. Circa il 30% delle fatture '
                            'è sbagliato. Dobbiamo inviare le fatture elettroniche entro domani.',
        'p_cliente':        'P3',
        'p_attesa':         'P2',
    },
    {
        'descrizione_test': 'P4: richiesta evolutiva',
        'oggetto':          'Richiesta aggiunta campo note in scheda paziente',
        'descrizione':      'Sarebbe comodo avere un campo note aggiuntivo nella '
                            'scheda anagrafica del paziente per inserire informazioni '
                            'aggiuntive non previste dai campi standard.',
        'p_cliente':        'P2',
        'p_attesa':         'P4',
    },
]

print('=' * 70)
for caso in CASI_TEST:
    pred = clf.predict(
        caso['oggetto'],
        caso['descrizione'],
        caso['p_cliente']
    )
    ok = '✓' if pred.priorita == caso['p_attesa'] else '✗'
    print(f"{ok} [{caso['descrizione_test']}]")
    print(f"   Attesa: {caso['p_attesa']} | Cliente: {caso['p_cliente']} | "
          f"Predetto: {pred.priorita} ({pred.confidenza:.0%}) via {pred.metodo.upper()}")
    if pred.reasoning:
        print(f"   Reasoning: {pred.reasoning}")
    print()

## STEP 9 — Valutazione su campione del test set

Carichiamo il dataset pulito e valutiamo il classificatore ibrido su un campione stratificato,
confrontando con il solo LinearSVC v2.

**Nota**: usa un campione piccolo (100-200 ticket) per il LLM — controlla i costi prima!

In [ ]:
from sklearn.metrics import classification_report, f1_score

# ─── Carica dataset pulito ────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'dataset_clean.csv', parse_dates=['data_creazione'])
print(f'Dataset: {len(df):,} righe')

# Filtro test temporale (stesso split usato in training)
SOGLIA_SPLIT = pd.Timestamp('2025-11-01')
df_test = df[df['data_creazione'] >= SOGLIA_SPLIT].copy()
print(f'Test set (dal {SOGLIA_SPLIT.date()}): {len(df_test):,} righe')
print('\nDistribuzione priorità nel test set:')
print(df_test['priorita_finale'].value_counts())

In [ ]:
# Campionamento stratificato (compatibile pandas 2.x)
# IMPORTANTE: bilancia il campione per avere abbastanza esempi di P4
N_PER_CLASSE = 30  # 30 * 4 classi = 120 ticket totali

campioni = pd.concat([
    df_test[df_test["priorita_finale"] == cls].sample(
        min(N_PER_CLASSE, (df_test["priorita_finale"] == cls).sum()),
        random_state=42
    )
    for cls in CLASSI
    if (df_test["priorita_finale"] == cls).sum() > 0
]).reset_index(drop=True)

print(f"Campione totale: {len(campioni)} ticket")
print(campioni["priorita_finale"].value_counts())

In [ ]:
# ─── Valutazione ─────────────────────────────────────────────────────────────
risultati = []
errori    = 0

for i, (_, row) in enumerate(campioni.iterrows()):
    try:
        pred = clf.predict(
            str(row.get('testo_input', '')),
            '',
            str(row.get('priorita_iniziale_cliente', 'P3'))
        )
        risultati.append({
            'priorita_reale':    row['priorita_finale'],
            'priorita_predetta': pred.priorita,
            'confidenza':        pred.confidenza,
            'metodo':            pred.metodo,
            'reasoning':         pred.reasoning or '',
            'corretto':          row['priorita_finale'] == pred.priorita,
        })

        if (i + 1) % 20 == 0:
            acc = sum(r['corretto'] for r in risultati) / len(risultati)
            n_llm = sum(1 for r in risultati if r['metodo'] == 'llm')
            print(f'[{i+1}/{len(campioni)}] Accuracy: {acc:.1%} | '
                  f'LLM usato: {n_llm}/{len(risultati)} ({n_llm/len(risultati):.0%})')

        # Rate limiting
        time.sleep(0.1)

    except Exception as e:
        errori += 1
        print(f'  Errore su idx {i}: {e}')

df_ris = pd.DataFrame(risultati)

print('\n' + '=' * 60)
print('RISULTATI CLASSIFICATORE IBRIDO v3')
print('=' * 60)
print(classification_report(
    df_ris['priorita_reale'],
    df_ris['priorita_predetta'],
    zero_division=0
))
print(f'Macro F1 ibrido:  {f1_score(df_ris["priorita_reale"], df_ris["priorita_predetta"], average="macro", zero_division=0):.4f}')
print(f'Macro F1 SVC v2:  {meta["macro_f1_test"]}  (benchmark)')
print(f'\nLLM usato:        {(df_ris["metodo"]=="llm").sum()} / {len(df_ris)} ticket '
      f'({(df_ris["metodo"]=="llm").mean():.0%})')
print(f'Errori API:       {errori}')

## STEP 10 — Analisi degli errori

Dove sbaglia ancora il classificatore ibrido?

In [ ]:
# ─── Analisi errori per classe ────────────────────────────────────────────────
errori_df = df_ris[~df_ris['corretto']].copy()
print(f'Errori totali: {len(errori_df)} / {len(df_ris)} '
      f'({len(errori_df)/len(df_ris):.1%})')
print('\nErrori per classe reale:')
print(errori_df['priorita_reale'].value_counts())

print('\nMatrice confusione:')
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    df_ris['priorita_reale'],
    df_ris['priorita_predetta'],
    labels=CLASSI
)
cm_df = pd.DataFrame(cm, index=CLASSI, columns=CLASSI)
cm_df.index.name = 'Reale \ Predetto'
print(cm_df)

In [ ]:
# ─── Dove ha aiutato l'LLM? ──────────────────────────────────────────────────
print('Performance SVC alone vs LLM alone (sui ticket che hanno passato l\'LLM):')

# Per i ticket dove il SVC era incerto e ha passato all'LLM:
df_llm_cases = df_ris[df_ris['metodo'] == 'llm'].copy()
if len(df_llm_cases) > 0:
    print(f'\nTicket gestiti da LLM: {len(df_llm_cases)}')
    print(f'Accuracy LLM su questi ticket: '
          f'{df_llm_cases["corretto"].mean():.1%}')
    print('\nDistribuzione classi reali nei casi LLM:')
    print(df_llm_cases['priorita_reale'].value_counts())

# ─── Esporta risultati ────────────────────────────────────────────────────────
output_path = OUT_DIR / 'hybrid_v3_risultati.csv'
df_ris.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'\nRisultati salvati in: {output_path}')

## STEP 11 — Prossimi step consigliati

### A. Migliorare il few-shot (priorità immediata)
Sostituire gli esempi manuali con esempi estratti automaticamente dal dataset,
usando il `decision_function` del LinearSVC per trovare i più prototipici per classe
(stesso approccio del `FewShot_ExampleExtractor.ipynb` per l'area).

```python
# Esempio: estrai i 3 ticket P4 con margine più alto
df_p4 = df_train[df_train['priorita_finale'] == 'P4']
X_p4 = build_features_batch(df_p4)  # da implementare
scores_p4 = svc.decision_function(X_p4)
idx_p4 = np.argsort(scores_p4[:, CLASSI.index('P4')])[-3:]
esempi_p4 = df_p4.iloc[idx_p4][['oggetto', 'descrizione_pulita', 'priorita_finale']]
```

### B. Calibrare la soglia di confidenza
Trovare la soglia ottimale con un validation set dedicato:
```python
soglie = np.arange(0.5, 0.95, 0.05)
# Per ogni soglia: valuta (costo_chiamate_LLM, macro_F1_risultante)
# Scegli la soglia che massimizza F1 con il minor numero di chiamate LLM
```

### C. Aggiungere la conversazione come feature
Il dataset completo (59k ticket) ha la `conversazione` con i messaggi cliente+operatore.
Aggiungere i primi 2 messaggi del cliente al testo input migliora il contesto per l'LLM.

### D. Endpoint FastAPI
```python
# Struttura endpoint suggerita
# POST /api/v1/classify/priority
# Body: {oggetto, descrizione, priorita_iniziale_cliente}
# Response: {priorita, confidenza, metodo, reasoning?}
```